In [1]:
import os

# Setting current working directory to where the notebook and run_pipeline.py are
os.chdir("D:/GoogleDrive/exercises/PPRL_applications_for_evaluation/personal_pprl_bloom_filters")

# Verifying
print("Current directory:", os.getcwd())

# Running the pipeline
%run run_pipeline.py

Current directory: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters


d:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Bloom Filter Based PPRL Pipeline...

Running: 00_synthetic_data_generation.ipynb


Executing: 100%|██████████| 25/25 [00:04<00:00,  6.01cell/s]



Running: 01_data_preparation.ipynb


Executing:  25%|██▌       | 2/8 [00:03<00:11,  1.85s/cell]


PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [1]":
---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[1], line 53
     49 # ========================================================
     50 # 5️⃣ Now you can use these paths in your notebook:
     51 # Example:
     52 SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
---> 53 PROCESSED_DATA_DIR = paths['PROCESSED_DATA_DIRs']
     54 # FIG_DIR = paths['FIG_DIR']
     55 
     56 # ========================================================

KeyError: 'PROCESSED_DATA_DIRs'


### Create Q-grams

In [ ]:
def get_qgrams(text, q=2):
    text = text.lower().replace(" ", "")
    return [text[i:i+q] for i in range(len(text) - q + 1)]

In [ ]:
get_qgrams("John")

### Creating a bloom filter

In [ ]:
import hashlib
import numpy as np

def hash_qgram(qgram, seed, size):
    return int(hashlib.md5((qgram + str(seed)).encode()).hexdigest(), 16) % size

In [ ]:
def create_bloom_filter(record, size=100, num_hashes=3):
    bf = np.zeros(size, dtype=int)

    # Combine attributes
    combined = record["name"] + record["dob"]
    qgrams = get_qgrams(combined)

    for qg in qgrams:
        for i in range(num_hashes):
            index = hash_qgram(qg, i, size)
            bf[index] = 1

    return bf

### Encode dataset

In [ ]:
bf_A = [create_bloom_filter(r) for r in records_A]
bf_B = [create_bloom_filter(r) for r in records_B]

### Compare bloom filters

In [ ]:
def dice_similarity(bf1, bf2):
    intersection = np.sum(bf1 * bf2)
    return (2 * intersection) / (np.sum(bf1) + np.sum(bf2))

In [ ]:
for i, a in enumerate(bf_A):
    for j, b in enumerate(bf_B):
        sim = dice_similarity(a, b)
        print(f"A{i} vs B{j}: {sim:.3f}")